# Read H-Reflex App Data Files

This notebook reads and visualizes data from the **H-Reflex Behavior App** (hreflex_txbdc) binary data files.

**File convention (V2/V3):**
- **`.hrs1`** — MH Recruitment Curve stage: sweeps across stimulation intensities to map the M/H-wave recruitment curve.
- **`.hrs2`** — Control Mode stage: stimulates at a fixed user-set intensity (can be changed between trials).
- **`.hrs3`** — Down Condition Pellet (DCP) stage: closed-loop H-reflex conditioning with pellet reward.
- **`.hrs4`** — Up Condition Pellet stage.
- **`.hrs5`** — Down Condition VNS stage.
- **`.hrs6`** — Up Condition VNS stage.
- **`.hrsft`** — Frequency Test stage.

For conditioning stages, **Section 3b** splits trials into **Rewarded vs Not rewarded** (VNS / pellet delivered).

All trial files share the MhRecHeader + MhRecTrial binary format.
EMG data blocks (raw differential, filtered, abs-value) are embedded in every file.

The binary format is based on the `FileIO_Helpers` serialization from the `hreflex_txbdc` package.

# Section 1: Binary File Reader Utilities

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from helpers import (
    # Multi-recording loading
    load_all_recordings, build_settings_dataframe,
    make_viewer,
    # Post-hoc global windowing
    analyze_global_background, run_threshold_sweep, plot_threshold_sweep,
    run_direct_bg_sweep, plot_direct_bg_sweep,
    # M/H response metrics (per-trial normalized)
    compute_mh_metrics, compute_mh_variability_sweep,
    plot_mh_variability, print_mh_variability_summary, view_mh_bin,
    # Section 5 quartile window comparison
    plot_quartile_window_comparison,
    # EMG window optimization
    compute_optimization_scores, plot_optimization, print_optimization_summary,

    plot_amplitude_distribution, plot_background_emg_views,
    plot_actual_trial_timeline,
    simulate_trial_initiation_hrs,
    split_trials_by_polarity, plot_hm_ratio_summary, plot_hwave_regression,
    plot_mwave_control_error, plot_frequency_test,

    # Conditioning / VNS reward analysis
    exclude_failed_trials, compute_reward_groups, get_trial_reward,
    get_applied_success_thresholds,
    plot_h_wave_reward_timeline, format_h_wave_reward_summary,
    plot_background_grand_means, plot_hrs2_trials,
    plot_bin_overview, create_bin_viewer, print_bin_statistics,
    compute_peri_stim_bg,
    # Constants
    SAMPLE_RATE, BIN_DURATION_MS, BIN_SAMPLES, TRIAL_RECORD_MS,
    STIM_ONSET_THRESHOLD, STIM_END_THRESHOLD,
)

print("Helpers loaded.")
print(f"  App constants: SAMPLE_RATE={SAMPLE_RATE} Hz | BIN={BIN_DURATION_MS} ms ({BIN_SAMPLES} samples) | TRIAL_RECORD={TRIAL_RECORD_MS} ms")
print(f"  Stim thresholds: onset >= {STIM_ONSET_THRESHOLD} V | end < {STIM_END_THRESHOLD} V")

# Section 1b: Multi-Recording Configuration

Configure one or more recordings in `RECORDING_DIRS` below (label, folder path,
sample rate — `None` to auto-detect). The `.hrs1`–`.hrs6` and `.hrft` files in
each folder are found automatically.

In [ ]:
# ── Multi-Recording Configuration ─────────────────────────────────────────────
# Each entry: ("Display Label", "relative/path/to/recording_dir", sample_rate_hz)
# sample_rate_hz: explicit Hz (e.g. 10000.0 or 5000.0); None = auto-detect from data.
RECORDING_DIRS = [
    # HRPilot-36 Up-VNS Conditioning Recordings
    ("UPVNS9 HRPILOT-36",   "Up-Conditioning_VNS/HRPilot-36/UPVNS9_HRPILOT-36_BOOTH2_250US_10KHZ_10-15-26",            10000.0),
    ("UPVNS10 HRPILOT-36",  "Up-Conditioning_VNS/HRPilot-36/UPVNS10_NEW_SETUP_HRPILOT-36_BOOTH2_250US_10KHZ_10-16-26",  10000.0),
    ("UPVNS11 HRPILOT-36",  "Up-Conditioning_VNS/HRPilot-36/UPVNS11_NEW_SET-UP_HRPILOT-36_BOOTH2_250US_10KHZ_10-17-26", 10000.0),
    ("UPVNS12 HRPILOT-36",  "Up-Conditioning_VNS/HRPilot-36/UPVNS12_HRPILOT-36_BOOTH2_250US_10KHZ_10-19-26",           10000.0),

    # HRPilot-33 Up-VNS Conditioning Recordings
    ("UPVNS1 HRPILOT-33",   "Up-Conditioning_VNS/HRPilot-33/UPVNS1_HRPILOT-33_BOOTH3_250US_10KHZ_10-19-26",            10000.0),

#("D3_HRPILOT-17_200US_6-1-26", "D3_HRPILOT-17_200US_6-1-26", None),

    # Add more recordings below — uncomment or append new tuples.
]
# When multiple recordings are loaded, pick which one to analyze with
# ACTIVE_RECORDING in the Recording & Stage Selector cell below.
print(f"{len(RECORDING_DIRS)} recording(s) configured.")
for _i, (_lbl, _rdir, _rsr) in enumerate(RECORDING_DIRS):
    _sr_str = f"{_rsr} Hz" if _rsr else "auto-detect"
    print(f"  [{_i}] {_lbl!r}  →  {_rdir}  (sample_rate={_sr_str})")

In [ ]:
# ── Load all recordings (auto-detects V2 / V3) ─────────────────────────────────
# _all_recordings_raw is never modified; the Failed-Trial Exclusion cell below
# replaces _all_recordings with a cleaned copy.
_all_recordings_raw = load_all_recordings(RECORDING_DIRS)
_all_recordings = _all_recordings_raw
_active_rec_label = next(iter(_all_recordings))
print(f'\n{len(_all_recordings)} recording(s) loaded.  Active: {_active_rec_label!r}')

### Failed-Trial Exclusion

Removes failed trials (no sync, alignment miss, recalculated bad stim ADC, frozen stim ADC / sync, onset mismatch) from every recording **before** the stage selector, using the same `exclude_failed_trials` as `Read_H-Reflex_App_Simplified.ipynb`. Every cell below uses the cleaned `_all_recordings`; the untouched load stays in `_all_recordings_raw`. `excluded_trials_df` lists one row per removed trial.

In [ ]:
# ── Failed-trial exclusion ────────────────────────────────────────────────────
EXCLUDE_FAILED_TRIALS = True
# 'all' = the Failed Trial Viewer's "All offline" group. Or list keys,
# e.g. ['no_sync', 'recalc_stim', 'align_miss']. App flags ('app_stim',
# 'app_sync', 'app_both', 'app_vns') are opt-in only (they over-flag).
EXCLUDE_CATEGORIES = 'all'
EXCLUDE_STAGES     = None     # None = all stages, or e.g. ['up_cond_vns']

# Always starts from the raw load, so re-running with new settings is safe.
_all_recordings    = _all_recordings_raw
excluded_trials_df = None
if EXCLUDE_FAILED_TRIALS:
    _all_recordings, excluded_trials_df = exclude_failed_trials(
        _all_recordings_raw, categories=EXCLUDE_CATEGORIES, stages=EXCLUDE_STAGES)
else:
    print('Failed-trial exclusion OFF — failed trials kept.')
excluded_trials_df

# Section 3: Peri-Stimulus Trials — Viewer Selection

Each viewer below has its own independent **Recording** and **Stage** dropdowns —
browse any loaded recording without affecting the others.

> For the post-hoc analysis pipeline (Sections 4–7), use the **Analysis Recording Selector**
> between Section 3 and Section 4 to set the active recording, then re-run Sections 4–7.

In [ ]:
# ── Recording & Stage Selector ─────────────────────────────────────────────────
from IPython.display import display as _disp

# ── Loaded recordings summary ─────────────────────────────────────────────────
print(f'{len(_all_recordings)} recording(s) loaded:')
for _rl, _rd in _all_recordings.items():
    print(f'  {_rl!r}  (App V{_rd["app_version"]}  |  {_rd["sample_rate"]} Hz)')
    for _sk, (_st, _sh, _se, _slbl) in _rd['stage_map'].items():
        print(f'    · {_slbl}: {len(_st)} trials')
print()

# ── Stimulation Intensity Histogram ──────────────────────────────────────────
def _render_hist(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Histogram: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_amplitude_distribution(trials, header)

_hist_widget, _hist_render = make_viewer(_all_recordings, _active_rec_label, _render_hist)
_disp(_hist_widget)
_hist_render()

In [ ]:
# ── Background EMG Views ──────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_bg(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Background EMG: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_background_emg_views(trials, emg_blocks, monitoring_window_ms=2500,
                              sample_rate=sr or h1h.sample_rate,
                              bin_duration_ms=h1h.bin_duration_ms)

_bg_widget, _bg_render = make_viewer(_all_recordings, _active_rec_label, _render_bg)
_disp(_bg_widget)
_bg_render()

In [ ]:
# ── Trial Timeline ─────────────────────────────────────────────────────────────
from IPython.display import display as _disp

def _render_tl(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    print(f'\n── Trial Timeline: {stage_label}  ({len(trials)} trials)  [{rec_label}]')
    plot_actual_trial_timeline(trials, header=header)

_tl_widget, _tl_render = make_viewer(_all_recordings, _active_rec_label, _render_tl)
_disp(_tl_widget)
_tl_render()

In [ ]:
# ── Analysis Recording Selector (Sections 4–7) ────────────────────────────────
# Sections 4–7 run on a single recording + stage at a time.
# Use the dropdowns below to pick the recording and stage, then re-run
# Sections 4–7 to update all downstream analysis.
from ipywidgets import Dropdown, VBox, Label
from IPython.display import display as _disp

def _ana_stage_opts(rec_label):
    sm = _all_recordings[rec_label]['stage_map']
    return [(lbl, sk) for sk, (_t, _h, _e, lbl) in sm.items() if _t]

_ana_rec_d = Dropdown(
    options=list(_all_recordings.keys()),
    value=_active_rec_label,
    description='Recording:', layout={'width': '640px'}
)
_ana_stage_opts_init = _ana_stage_opts(_active_rec_label)
_ana_stage_d = Dropdown(
    options=_ana_stage_opts_init,
    value=_ana_stage_opts_init[0][1] if _ana_stage_opts_init else None,
    description='Stage:', layout={'width': '480px'}
)
_ana_info_lbl = Label(value='')

def _update_analysis_globals():
    global _plot_trials, _plot_header, _plot_emg_blocks, recording_sample_rate, hrs1_header
    _rl = _ana_rec_d.value
    _sk = _ana_stage_d.value
    if not _rl or not _sk or _sk not in _all_recordings[_rl]['stage_map']:
        return
    _rec = _all_recordings[_rl]
    _st, _sh, _se, _slbl = _rec['stage_map'][_sk]
    _plot_trials, _plot_header, _plot_emg_blocks = _st, _sh, _se
    recording_sample_rate = _rec['sample_rate']
    hrs1_header = _rec['hrs1_header']
    _ana_info_lbl.value = (
        f'Active: {_rl!r}  →  {_slbl}  ({len(_plot_trials)} trials)  '
        f'— re-run Sections 4–7 below to update the analysis.'
    )

def _on_ana_rec(change):
    _ana_stage_d.unobserve(_on_ana_stage, names='value')
    opts = _ana_stage_opts(_ana_rec_d.value)
    _ana_stage_d.options = opts
    _ana_stage_d.value = opts[0][1] if opts else None
    _ana_stage_d.observe(_on_ana_stage, names='value')
    _update_analysis_globals()

def _on_ana_stage(change):
    _update_analysis_globals()

_ana_rec_d.observe(_on_ana_rec, names='value')
_ana_stage_d.observe(_on_ana_stage, names='value')

_disp(VBox([_ana_rec_d, _ana_stage_d, _ana_info_lbl]) if len(_all_recordings) > 1
      else VBox([_ana_stage_d, _ana_info_lbl]))
_update_analysis_globals()

# Section 3b: Conditioning — Rewarded vs Not Rewarded

Conditioning stages only (.hrs3–.hrs6). A trial is **rewarded** when the app saved `vns_delivered` / `pellet_delivered` = 1 (the app *commanded* the reward TTL).

1. **Reward timeline**: H-wave size per trial, red = rewarded, green dotted = the success threshold the app applied. Has its own Recording + Stage dropdowns.
2. **Rewarded vs Not rewarded**: the Bin Overview, Bin Statistics and Per-Bin Viewer, with the two reward groups in place of EMG bins. Runs on `_plot_trials` (the selected recording + stage).
3. **VNS Trial Viewer**: only the VNS-delivered trials, with VNS ADC / VNS Sync rows to check delivery.
4. **`reward_df`**: per-trial metrics with a `rewarded` column, and `reward_summary` (count / mean / SD per group).

Uses the recording + stage chosen in the **Analysis Recording Selector** above; re-run this section after changing it.

In [ ]:
# ── Section 3b configuration ─────────────────────────────────────────────────
# Viewing windows (ms relative to stim onset)
PRE_PLOT_MS, POST_PLOT_MS = 2, 50
PRE_AVG_MS,  POST_AVG_MS  = 2, 15
N_PER_PAGE = 6

# M/H windows for Section 3b. Defaults = the app's own windows in the HRPilot-36
# UPVNS9–12 settings.json (HRPilot-33 UPVNS1 used H 6–10 ms). Section 6a below
# sets its own M_START_MS / H_START_MS independently.
M_WAVE_START_MS, M_WAVE_END_MS = 2.0, 4.0
H_WAVE_START_MS, H_WAVE_END_MS = 5.5, 9.5

_sr_3b = recording_sample_rate or hrs1_header.sample_rate
print(f"M-wave {M_WAVE_START_MS}–{M_WAVE_END_MS} ms  |  H-wave {H_WAVE_START_MS}–{H_WAVE_END_MS} ms")

# Per-trial stored background grand mean (used by the per-group viewer).
reward_state = plot_background_grand_means(_plot_trials, _plot_emg_blocks, _plot_header,
                                           sample_rate=_sr_3b)

In [ ]:
# ── H-wave Size per Trial: Conditioning Reward Timeline ────────────────────────
# Grey = no reward, red = reward delivered, green dotted = success threshold the
# app applied to that trial. Same viewer as Read_H-Reflex_App_Simplified.ipynb.
from ipywidgets import ToggleButtons, VBox
from IPython.display import display as _disp

def _hrt_stage_filter(key, trials, header, emg_blocks, label):
    return bool(trials) and get_trial_reward(trials[0])[1] is not None

_hrt_src_tog = ToggleButtons(
    options=[('In-session (app)', 'in_session'), ('Offline window', 'offline')],
    value='in_session', description='H-wave:',
)
_hrt_scope_tog = ToggleButtons(
    options=[('Selected recording', 'one'), ('All recordings', 'all')],
    value='one', description='Scope:',
)

_has_any_hrt = any(
    any(_hrt_stage_filter(sk, _t, _h, _e, lbl)
        for sk, (_t, _h, _e, lbl) in rd['stage_map'].items())
    for rd in _all_recordings.values()
)

if _has_any_hrt:
    def _render_hrt(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
        if _hrt_scope_tog.value == 'all':
            _sessions = [(_rl, _t, _h, _rd['sample_rate'] or _rd['hrs1_header'].sample_rate)
                         for _rl, _rd in _all_recordings.items()
                         for _sk, (_t, _h, _e, _lbl) in _rd['stage_map'].items()
                         if _lbl == stage_label and _t]
        else:
            _sessions = [(rec_label, trials, header, sr or h1h.sample_rate)]
        print(f'\n── H-wave Reward Timeline: {stage_label}  '
              f'({len(_sessions)} session(s))  [{", ".join(s[0] for s in _sessions)}]')
        _hrt_summary = plot_h_wave_reward_timeline(
            _sessions, source=_hrt_src_tog.value,
            h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
            pre_ms=PRE_AVG_MS, post_ms=POST_AVG_MS,
        )
        if _hrt_summary:
            print(format_h_wave_reward_summary(_hrt_summary, get_trial_reward(_sessions[0][1][0])[1]))

    _hrt_widget, _hrt_render = make_viewer(
        _all_recordings, _active_rec_label, _render_hrt,
        stage_filter=_hrt_stage_filter,
    )
    for _w in (_hrt_src_tog, _hrt_scope_tog):
        _w.observe(lambda c: _hrt_render(), names='value')
    _disp(VBox([_hrt_src_tog, _hrt_scope_tog, _hrt_widget]))
    _hrt_render()
else:
    print("No conditioning-stage recordings (.hrs3–.hrs6) loaded.")

In [ ]:
# ── Rewarded vs Not Rewarded: group the selected stage's trials ───────────────
reward_groups = compute_reward_groups(_plot_trials, reward_state)
if reward_groups:
    (reward_binned, reward_labels, reward_colors,
     reward_bg_dict, reward_pol_labels, _) = reward_groups

In [ ]:
# ── Rewarded vs Not Rewarded: averaged waveforms + M / H / H:M bars ───────────
if reward_groups:
    plot_bin_overview(
        reward_binned, reward_labels, reward_colors, reward_pol_labels, _plot_header,
        sample_rate=_sr_3b,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    )

In [ ]:
# ── Rewarded vs Not Rewarded: numeric table ──────────────────────────────────
if reward_groups:
    print_bin_statistics(
        reward_binned, reward_labels, reward_pol_labels,
        sample_rate=_sr_3b,
        pre_avg_ms=PRE_AVG_MS, post_avg_ms=POST_AVG_MS,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    )

In [ ]:
# ── Rewarded vs Not Rewarded: per-group viewer (pick group, click Load) ───────
if reward_groups:
    create_bin_viewer(
        reward_binned, reward_labels, reward_colors, reward_pol_labels,
        _plot_header, _plot_emg_blocks, reward_bg_dict,
        sample_rate=_sr_3b,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        pre_avg_ms=PRE_AVG_MS,   post_avg_ms=POST_AVG_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
    )

In [ ]:
# ── VNS Trial Viewer: per-trial grid + zoom, VNS-delivered trials only ─────────
# Same viewer as Read_H-Reflex_App_Simplified.ipynb. Panel titles keep each
# trial's original session number. Show VNS ADC / Show VNS Sync DIG IN add rows
# on their own time axis (whole stored VNS buffer, ~585 ms after onset).
from IPython.display import display as _disp

VNS_THRESHOLD_V = 0.5   # |VNS ADC − pre-trigger baseline| (V) that counts as the pulse

def _render_vns_trv(trials, header, emg_blocks, stage_label, rec_label, sr, h1h):
    _nums = {id(t): i + 1 for i, t in enumerate(trials)}
    _vns_trials = [t for t in trials if getattr(t, 'vns_delivered', 0)]
    print(f'\n── VNS Trial Viewer: {stage_label}  '
          f'({len(_vns_trials)} VNS of {len(trials)} trials)  [{rec_label}]')
    if not _vns_trials:
        print('No VNS-delivered trials in this stage.')
        return
    plot_hrs2_trials(
        _vns_trials, header,
        pre_plot_ms=PRE_PLOT_MS, post_plot_ms=POST_PLOT_MS,
        n_per_page=N_PER_PAGE,
        m_start_ms=M_WAVE_START_MS, m_end_ms=M_WAVE_END_MS,
        h_start_ms=H_WAVE_START_MS, h_end_ms=H_WAVE_END_MS,
        sample_rate=sr or h1h.sample_rate,
        emg_blocks=emg_blocks,
        trial_numbers=_nums,
        vns_threshold_v=VNS_THRESHOLD_V,
    )

_vns_trv_widget, _vns_trv_render = make_viewer(
    _all_recordings, _active_rec_label, _render_vns_trv,
    stage_filter=lambda sk, trials, *_a: bool(trials) and hasattr(trials[0], 'vns_delivered'),
)
_disp(_vns_trv_widget)
_vns_trv_render()

In [ ]:
# ── Per-trial reward DataFrame + Rewarded vs Not Rewarded summary ─────────────
import pandas as pd
from helpers import get_trial_window as _gtw_rw

_sr_rw  = _sr_3b
# Applied threshold comes from the full (pre-exclusion) session, because the app's
# threshold for trial i is saved on trial i + 1 (see get_applied_success_thresholds).
_raw_rw = _all_recordings_raw[_ana_rec_d.value]['stage_map'][_ana_stage_d.value][0]
_thr_rw = dict(zip(map(id, _raw_rw), get_applied_success_thresholds(_raw_rw)))
_num_rw = {id(t): i + 1 for i, t in enumerate(_raw_rw)}
_rows = []
for _t in _plot_trials:
    _tm, _emg, _, _, _ = _gtw_rw(_t, PRE_AVG_MS, POST_AVG_MS, ms_per_sample=1000.0 / _sr_rw)
    _mm = (_tm >= M_WAVE_START_MS) & (_tm <= M_WAVE_END_MS)
    _hm = (_tm >= H_WAVE_START_MS) & (_tm <= H_WAVE_END_MS)
    _bg = compute_peri_stim_bg(_t, _sr_rw)
    _bg = 0.0 if np.isnan(_bg) else _bg
    _mv = float(np.nanmean(np.abs(_emg[_mm]))) - _bg if _mm.any() else float('nan')
    _hv = float(np.nanmean(np.abs(_emg[_hm]))) - _bg if _hm.any() else float('nan')
    _rows.append({
        'session_trial':     _num_rw.get(id(_t), -1),   # 1-based number in the raw session
        'rewarded':          get_trial_reward(_t)[0],
        'stim_amp_ma':       float(getattr(_t, 'stimulation_amplitude_ma', float('nan'))),
        'bg_mra_uv':         _bg,
        'm_size_uv':         _mv,
        'h_size_uv':         _hv,
        'hm_ratio':          _hv / _mv if _mv > 0 else float('nan'),
        'app_h_wave_uv':     float(getattr(_t, 'h_wave_response', float('nan'))),
        'applied_threshold': float(_thr_rw.get(id(_t), float('nan'))),
    })
reward_df = pd.DataFrame(_rows)
reward_summary = (reward_df.groupby('rewarded')
                  [['m_size_uv', 'h_size_uv', 'hm_ratio', 'bg_mra_uv', 'app_h_wave_uv']]
                  .agg(['count', 'mean', 'std']).round(3))
print(f"reward_df: {len(reward_df)} trials  |  rewarded: {int(reward_df['rewarded'].sum())}")
reward_summary

# Section 4: Post-Hoc Background EMG Analysis

Reconstruct the continuous abs-EMG signal from HRS2 EMG blocks, blank around each stimulation event, compute background statistics, extract per-trial pre-stim grand means, and run the direct background sweep.

**Steps:**
1. **4a** — Sort and stitch all HRS2 EMG blocks into one continuous signal.
2. **4b** — Build a blank mask that suppresses samples within ±[`blank_pre_ms`, `blank_post_ms`] of every stimulation event.
3. **4c** — Compute global background EMG statistics from the non-blanked signal.
4. **4d** — Extract each trial's pre-stim background grand mean and plot the distribution.
5. **4e** — Run the direct per-trial background sweep (EMG window width vs matched trials).

In [ ]:
# ---- Section 4: Post-hoc background EMG analysis ----
state = analyze_global_background(
    _plot_trials, _plot_emg_blocks, _plot_header,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    blank_pre_ms=5, blank_post_ms=20,
    min_valid_frac=0.7,
)

# Section 4b: Virtual Trial Rate Sweep

Walks the continuous background EMG signal (with stim periods blanked) using the
same state machine as the H-Reflex app: random monitoring windows 2200–2700 ms,
50 ms bins, accepts when all bin grand means fall within the threshold window.

**Output**: virtual trial rate (trials/hour) vs EMG window half-width,
evaluated at Q1, Median, and Q3 of the background distribution.

This is the primary sweep used by the **Section 7 Optimization**.

In [ ]:
# ---- Section 4b: Virtual trial rate sweep ----
# sweep_centres and half_widths_uv are reused by the M-H variability sweep
# in Section 6a -- keep them consistent for Section 7 optimization.
sweep_centres  = {'Q1': state['gm_q1'], 'Median': state['gm_med'], 'Q3': state['gm_q3']}
half_widths_uv = [5, 10, 20, 30, 45, 60, 75, 100, 125, 150]

sweep_results = run_threshold_sweep(
    state, sweep_centres=sweep_centres, half_widths_uv=half_widths_uv,
)
plot_threshold_sweep(sweep_results, state, _plot_trials, _plot_header)

# Section 4c: Direct Background Trial Count (Reference)

Retrospective count: of the **actual HRS2 trials recorded**, how many had a
pre-stimulus background grand mean within each threshold window?

> **Note**: this does *not* simulate trial rate — it only counts trials that
> *were already collected* whose background happened to be in-window.
> For prospective trial rate estimation, see Section 4b above.

In [ ]:
# ---- Section 4c: Direct background trial count (retrospective reference) ----
direct_bg_results = run_direct_bg_sweep(state)
plot_direct_bg_sweep(direct_bg_results, state, _plot_trials, _plot_header)

# Section 4d: Trial Initiation Simulator

Re-runs the H-Reflex app's trial-initiation state machine on the recorded EMG signal.

Uses peri-stim EMG blocks from the current stage; samples around each stim onset are zeroed so stim artifacts cannot trigger initiation.

Set `SIM_MIN_UV`, `SIM_MAX_UV`, and `SIM_MIN_ITI_MS` in the cell below, then run it.

In [ ]:
# ---- Section 4d: Trial Initiation Simulation ----
# Re-runs the H-Reflex app's trial-initiation state machine on the recorded EMG signal.
# Uses peri-stim EMG blocks from the current stage; samples around each stim onset are
# zeroed so stim artifacts cannot trigger initiation.

# Initiation thresholds (µV). Set to None to read from recording header.
SIM_MIN_UV      = None     # e.g. 50.0  or None
SIM_MAX_UV      = None     # e.g. 200.0 or None

SIM_MIN_ITI_MS    = 3000
SIM_BLANK_PRE_MS  = 5.0
SIM_BLANK_POST_MS = 20.0

_sim_min = SIM_MIN_UV if SIM_MIN_UV is not None else float(getattr(hrs1_header, 'trial_initiation_uv_min', 5.0))
_sim_max = SIM_MAX_UV if SIM_MAX_UV is not None else float(getattr(hrs1_header, 'trial_initiation_uv_max', 300.0))

simulated_trials, sim_statistics = simulate_trial_initiation_hrs(
    _plot_emg_blocks,
    _plot_header,
    hrs2_trials=_plot_trials,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
    min_init_uv=_sim_min,
    max_init_uv=_sim_max,
    min_inter_trial_ms=SIM_MIN_ITI_MS,
    blank_pre_ms=SIM_BLANK_PRE_MS,
    blank_post_ms=SIM_BLANK_POST_MS,
)

# Section 5: Quartile Window Comparison

For a fixed **EMG window width** (±`hw_uv` µV), compare the trials selected by windows
centred on **Q1**, **Median**, and **Q3** of the background distribution.

- **Plot 1** — histogram of per-trial background with shaded EMG windows for each quartile.
- **Interactive viewer** — dropdown selects Q1 / Median / Q3 / All:
  - *Left*: peri-stimulus waveforms (−2 ms to 15 ms), M/H window shading,
    group avg (black) + SEM, overall recording avg (blue).
  - *Right*: 2500 ms pre-stimulus background, same group vs overall overlays.

In [ ]:
# ---- Section 5: Quartile window comparison (fixed EMG window width) ----
M_START_MS_S5, M_END_MS_S5 = 2.5, 5.5
H_START_MS_S5, H_END_MS_S5 = 7, 11.5

plot_quartile_window_comparison(
    _plot_trials, _plot_emg_blocks, state, _plot_header,
    hw_uv=50,
    pre_ms=2.0, post_ms=15.0,
    bg_pre_ms=2500.0,
    m_start_ms=M_START_MS_S5, m_end_ms=M_END_MS_S5,
    h_start_ms=H_START_MS_S5, h_end_ms=H_END_MS_S5,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 6: M-/H-Response Variability vs EMG Window Size

For every trial we measure three response metrics inside the M and H windows,
each normalised to that trial's own pre-stim background grand mean:

- **M-H Peak (MRA)** — `mean(|emg|)` in the M or H window  (µV)
- **H/M Ratio** — `H-MRA / M-MRA` per trial
- **M-H Size** — `(MRA − per_trial_bg) / per_trial_bg`  (units of background activity)

For centres at **Q1 / Median / Q3** of the background distribution we sweep a
EMG window width `hw`, group trials whose background falls in `[centre ± hw]`, and
compute STD and CV (Coefficient of Variation = STD / Mean) for all metrics.

Output:
1. A 3 × 2 figure — STD (left) and CV (right) for M-H Peak, H/M Ratio, M-H Size.
2. A summary table at the 10 marker EMG window widths.
3. An interactive bin viewer.

In [ ]:
# ---- Section 6a: Per-trial M-H metrics (per-trial background normalization) ----
M_START_MS, M_END_MS = 3.0, 5.5
H_START_MS, H_END_MS = 6.5, 11.5

mh_metrics = compute_mh_metrics(
    _plot_trials, state['trial_bg_gm'],
    m_start_ms=M_START_MS, m_end_ms=M_END_MS,
    h_start_ms=H_START_MS, h_end_ms=H_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

sweep_centres  = {'Q1': state['gm_q1'], 'Median': state['gm_med'], 'Q3': state['gm_q3']}
half_widths_uv = [5, 10, 20, 30, 45, 60, 75, 100, 125, 150]

mh_variability = compute_mh_variability_sweep(
    state['trial_bg_gm'], mh_metrics, sweep_centres, half_widths_uv,
)

n = len(_plot_trials)
print(f"Computed M-H metrics for {n} trials (per-trial background normalization)")
print(f"M window: {M_START_MS}–{M_END_MS} ms  |  H window: {H_START_MS}–{H_END_MS} ms")
print()
for key, label in [('m_mra',    'M-MRA (µV)     '),
                   ('h_mra',    'H-MRA (µV)     '),
                   ('hm_ratio', 'H/M ratio       '),
                   ('m_size',   'M-size (×bg)   '),
                   ('h_size',   'H-size (×bg)   ')]:
    arr = mh_metrics[key]
    v = arr[~np.isnan(arr)]
    if len(v) == 0:
        print(f"  {label}: no valid data")
        continue
    print(f"  {label}  n={len(v)}  mean={np.mean(v):.4f}  "
          f"std={np.std(v, ddof=1):.4f}  median={np.median(v):.4f}  "
          f"min={np.min(v):.4f}  max={np.max(v):.4f}")


In [ ]:
# ---- Section 6b: Variability plot (3 rows x 2 cols: STD / CV) ----
marker_hws = plot_mh_variability(mh_variability, _plot_header, marker_hws=half_widths_uv)
print(f"Marker EMG window widths: {[f'{hw:.1f}' for hw in marker_hws]}")

In [ ]:
# ---- Section 6c: Summary table at the 10 marker EMG window widths ----
mh_at_markers = compute_mh_variability_sweep(
    state['trial_bg_gm'], mh_metrics, sweep_centres,
    half_widths_uv=marker_hws,
)
print_mh_variability_summary(mh_at_markers, sweep_centres, _plot_header,
                              marker_hws=marker_hws)

# Section 6d: Manual EMG Activity Bins

Classify trials into user-defined pre-stim EMG level bins using the per-trial grand means from Section 4 (`state['trial_bg_gm']`).

Edit the `BIN_LO_x / BIN_HI_x` pairs below to set your own µV ranges.
Trials whose background grand mean falls within `[lo, hi)` are grouped into that bin.

**Outputs:**
- **6d-bins** — bin definition + trial count summary
- **6d-overview** — averaged waveform overlay + M / H / H:M bar charts + stats panel per polarity


In [ ]:
# ---- Section 6d-bins: Define EMG Activity Bins ----
BIN_LO_1, BIN_HI_1 =  50, 150   # µV  Bin 1
BIN_LO_2, BIN_HI_2 = 150, 200   # µV  Bin 2
BIN_LO_3, BIN_HI_3 = 200, 350   # µV  Bin 3

_EMG_BINS   = [(BIN_LO_1, BIN_HI_1), (BIN_LO_2, BIN_HI_2), (BIN_LO_3, BIN_HI_3)]
_BIN_LABELS = [f'Bin {i+1}: {lo}–{hi} µV' for i, (lo, hi) in enumerate(_EMG_BINS)]
_BIN_COLORS = ['steelblue', 'darkorange', 'forestgreen', 'crimson', 'mediumpurple'][:len(_EMG_BINS)]

_bg_gm    = state['trial_bg_gm']
_by_pol   = split_trials_by_polarity(_plot_trials)
_pol_lbls = list(_by_pol.keys())
_id_map   = {id(t): i for i, t in enumerate(_plot_trials)}

_binned = {
    pol: {
        lbl: [t for t in trs if lo <= _bg_gm[_id_map[id(t)]] < hi]
        for lbl, (lo, hi) in zip(_BIN_LABELS, _EMG_BINS)
    }
    for pol, trs in _by_pol.items()
}

_hdr = f"{'Bin':<25}  " + "  ".join(f"{p:<12}" for p in _pol_lbls)
print(_hdr)
print("-" * len(_hdr))
for lbl in _BIN_LABELS:
    _row = "  ".join(f"{len(_binned[p][lbl]):<12}" for p in _pol_lbls)
    print(f"{lbl:<25}  {_row}")


In [ ]:
# ---- Section 6d-overview: Combined EMG Bin Overview ----
# Averaged waveforms + M / H / H:M bar charts (MRA, mean ± STD) + stats panel.
# Polarity toggle when both polarities are present.
from helpers import get_trial_window as _gtw
from ipywidgets import ToggleButtons as _TB6, Output as _Out6, VBox as _VB6
from IPython.display import display as _disp6

_sr6  = recording_sample_rate or hrs1_header.sample_rate
_ms6  = 1000.0 / _sr6
_pre6, _post6 = 2.0, 15.0   # ms window around stim onset for averaged view
_out6 = _Out6()

def _draw_overview6(pol_lbl):
    _bins = _binned[pol_lbl]
    with _out6:
        _out6.clear_output(wait=True)

        _avgs, _m_mra, _h_mra, _hm_mra, _ns = {}, {}, {}, {}, {}
        for lbl in _BIN_LABELS:
            trs = _bins[lbl]
            _ns[lbl] = len(trs)
            if not trs:
                _avgs[lbl] = None
                _m_mra[lbl] = _h_mra[lbl] = _hm_mra[lbl] = []
                continue
            stacks, mv_l, hv_l, hm_l = [], [], [], []
            t_ref = None
            for tr in trs:
                _tm, _emg, _, _, _ = _gtw(tr, _pre6, _post6, ms_per_sample=_ms6)
                if t_ref is None:
                    t_ref = _tm
                stacks.append(_emg[:len(t_ref)])
                _mm = (_tm >= M_START_MS) & (_tm <= M_END_MS)
                _hm = (_tm >= H_START_MS) & (_tm <= H_END_MS)
                if _mm.any():
                    mv_l.append(float(np.nanmean(np.abs(_emg[_mm]))))
                if _hm.any():
                    hv_l.append(float(np.nanmean(np.abs(_emg[_hm]))))
                if _mm.any() and _hm.any():
                    mv = float(np.nanmean(np.abs(_emg[_mm])))
                    hv = float(np.nanmean(np.abs(_emg[_hm])))
                    if mv > 0:
                        hm_l.append(hv / mv)
            _avgs[lbl]  = (t_ref, np.nanmean(np.vstack(stacks), axis=0)) if stacks else None
            _m_mra[lbl] = mv_l
            _h_mra[lbl] = hv_l
            _hm_mra[lbl] = hm_l

        present = [b for b in _BIN_LABELS if _ns[b] > 0]
        cols    = [_BIN_COLORS[_BIN_LABELS.index(b)] for b in present]
        x_pos   = np.arange(len(present))

        fig, axes = plt.subplots(1, 5, figsize=(22, 5),
                                 gridspec_kw={'width_ratios': [5, 3, 3, 3, 2.5]})
        ax_w, ax_m, ax_h, ax_hm, ax_txt = axes

        for lbl, col in zip(present, cols):
            if _avgs[lbl] is not None:
                t_r, avg = _avgs[lbl]
                ax_w.plot(t_r, avg, color=col, linewidth=2.2,
                          label=f'{lbl}  (n={_ns[lbl]})')
        ax_w.axvspan(M_START_MS, M_END_MS, color='blue',  alpha=0.12, zorder=0)
        ax_w.axvspan(H_START_MS, H_END_MS, color='green', alpha=0.12, zorder=0)
        ax_w.axvline(0, color='red', linestyle='--', linewidth=1.0, label='Stim onset')
        ax_w.set_xlabel('Time re: stim onset (ms)')
        ax_w.set_ylabel('EMG (µV)')
        ax_w.set_title(f'Avg Waveforms — {pol_lbl}')
        ax_w.legend(fontsize=8, loc='upper right')
        ax_w.grid(True, alpha=0.3)

        def _bar6(ax, vd, ylabel, title):
            for xi, (lbl, col) in enumerate(zip(present, cols)):
                v = np.array([x for x in vd[lbl] if np.isfinite(x)])
                mu = float(np.mean(v)) if len(v) else 0.0
                sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
                ax.bar(xi, mu, color=col, alpha=0.78, yerr=sd, capsize=6,
                       error_kw={'elinewidth': 2, 'ecolor': col, 'capthick': 2})
            ax.set_xticks(x_pos)
            ax.set_xticklabels(present, rotation=20, ha='right', fontsize=7)
            ax.set_ylabel(ylabel, fontsize=9)
            ax.set_title(title, fontsize=10, fontweight='bold')
            ax.grid(True, axis='y', alpha=0.3)

        _bar6(ax_m,  _m_mra,  'MRA (µV)', 'M-wave')
        _bar6(ax_h,  _h_mra,  'MRA (µV)', 'H-wave')
        _bar6(ax_hm, _hm_mra, 'H:M',      'H:M Ratio')

        ax_txt.axis('off')
        lines = [f'Stats — {pol_lbl}', '']
        for lbl in present:
            lines.append(lbl)
            for key, vd in [('M', _m_mra), ('H', _h_mra), ('H:M', _hm_mra)]:
                v = np.array([x for x in vd[lbl] if np.isfinite(x)])
                if len(v):
                    mu = float(np.mean(v))
                    sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
                    cv = sd / mu if mu != 0 else float('nan')
                    lines.append(f'  {key}: n={len(v)} SD={sd:.2f} CV={cv:.2f}')
            lines.append('')
        ax_txt.text(0.05, 0.97, '\n'.join(lines), transform=ax_txt.transAxes,
                    va='top', ha='left', fontsize=7.5, family='monospace',
                    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='#cccccc'))

        plt.suptitle(f'Combined EMG Bin Overview  —  {_plot_header.subject_id}', fontsize=11)
        plt.tight_layout()
        plt.show()

if len(_pol_lbls) > 1:
    _tgl6 = _TB6(options=_pol_lbls, button_style='info', description='Polarity:')
    def _on_pol6(change):
        if change['name'] == 'value':
            _draw_overview6(change['new'])
    _tgl6.observe(_on_pol6, names='value')
    _disp6(_VB6([_tgl6, _out6]))
else:
    _disp6(_out6)
_draw_overview6(_pol_lbls[0])


In [ ]:
# ---- Section 6d: Interactive bin viewer ----
view_mh_bin(
    mh_at_markers, marker_hws, _plot_trials, mh_metrics, _plot_header,
    m_start_ms=M_START_MS, m_end_ms=M_END_MS,
    h_start_ms=H_START_MS, h_end_ms=H_END_MS,
    emg_blocks=_plot_emg_blocks,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)


# Section 6e: Per-Trial DataFrame & Calculations

`trial_df` is a flat per-trial DataFrame built from `mh_metrics` (Section 6a).  
`m_size` / `h_size` are normalized to background: `(MRA − BG) / BG` (dimensionless).  
Re-run if you re-run Section 6a with different windows.

In [ ]:
import pandas as pd

# ── Build per-trial DataFrame from mh_metrics ─────────────────────────────────
_bg_gm = state['trial_bg_gm']
_rows = []
for _i, _t in enumerate(_plot_trials):
    _rows.append({
        'trial':       _i + 1,
        'stim_amp_ma': float(getattr(_t, 'stimulation_amplitude_ma', float('nan'))),
        'bg_gm_uv':    float(_bg_gm[_i])                      if _i < len(_bg_gm)                     else float('nan'),
        'm_mra_uv':    float(mh_metrics['m_mra'][_i])          if _i < len(mh_metrics['m_mra'])         else float('nan'),
        'h_mra_uv':    float(mh_metrics['h_mra'][_i])          if _i < len(mh_metrics['h_mra'])         else float('nan'),
        'hm_ratio':    float(mh_metrics['hm_ratio'][_i])       if _i < len(mh_metrics['hm_ratio'])      else float('nan'),
        'm_size_norm': float(mh_metrics['m_size'][_i])         if _i < len(mh_metrics['m_size'])        else float('nan'),
        'h_size_norm': float(mh_metrics['h_size'][_i])         if _i < len(mh_metrics['h_size'])        else float('nan'),
    })

trial_df = pd.DataFrame(_rows)
print(f"trial_df: {len(trial_df)} rows  (m/h_size_norm = (MRA−BG)/BG, dimensionless)")
trial_df.head(10)

In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
trial_summary = pd.Series({
    'n_trials':      trial_df['trial'].count(),
    'm_mra_mean':    trial_df['m_mra_uv'].mean(),
    'm_mra_std':     trial_df['m_mra_uv'].std(),
    'h_mra_mean':    trial_df['h_mra_uv'].mean(),
    'h_mra_std':     trial_df['h_mra_uv'].std(),
    'hm_ratio_mean': trial_df['hm_ratio'].mean(),
    'hm_ratio_std':  trial_df['hm_ratio'].std(),
    'm_size_mean':   trial_df['m_size_norm'].mean(),
    'h_size_mean':   trial_df['h_size_norm'].mean(),
}, name='value').round(4).to_frame()
trial_summary

In [ ]:
# ── Custom calculations — edit this cell freely ───────────────────────────────
# Examples:

# Filter by background EMG level
# low_bg = trial_df[trial_df['bg_gm_uv'] < 100]

# Coefficient of variation (%) for H-MRA
# cv_h = trial_df['h_mra_uv'].std() / trial_df['h_mra_uv'].mean() * 100
# print(f"H-wave MRA CV: {cv_h:.1f}%")

# Correlation between background EMG and H-wave MRA
# trial_df[['bg_gm_uv', 'h_mra_uv']].corr()

# Export to CSV
# trial_df.to_csv('trial_data.csv', index=False)

print("trial_df and trial_summary are ready.  Edit this cell to run your calculations.")

### Recording Settings DataFrame

`settings_df` captures the app-side stimulation/window settings that were active for each
(recording, stage) — parsed from the `*.settings.json` sidecar written next to each binary
data file by newer H-Reflex App recordings (sweep range, M/H-wave windows, thresholds,
polarity, pulse-train parameters, etc). Stages recorded before the app wrote sidecars still
get a row here; their settings columns are simply NaN.

In [ ]:
# ── Build settings DataFrame from *.settings.json sidecars ───────────────────────
settings_df = build_settings_dataframe(_all_recordings)
_meta_cols = {'recording', 'stage', 'stage_key', 'subject_id', 'session_start',
              'app_version', 'file_version', 'sample_rate_hz', 'n_trials'}
_setting_cols = [c for c in settings_df.columns if c not in _meta_cols]
_n_with_settings = int(settings_df[_setting_cols].notna().any(axis=1).sum()) if _setting_cols else 0
print(f"settings_df: {len(settings_df)} rows  |  {settings_df['recording'].nunique()} recording(s)  |  "
      f"{_n_with_settings}/{len(settings_df)} stage(s) have a settings.json sidecar")
settings_df.head(10)

### Save Analysis to Disk

Bundle `trial_df`, `trial_summary`, and `settings_df` into a single pickle file so this analysis
can be reloaded later without re-reading the raw binary recordings. Call
`save_analysis_pickle()` whenever you want a snapshot — nothing is saved automatically.

In [ ]:
# ── Save analysis DataFrames to a pickle file (on demand) ──────────────
import pickle
from datetime import datetime as _dt

def save_analysis_pickle(path=None, **extra_frames):
    """Pickle trial_df, trial_summary, and settings_df (plus any extra_frames) to *path*.

    path=None auto-names the file from the current timestamp in the working directory.
    Returns the path written to.
    """
    _path = path or f"post_hoc_analysis_{_dt.now():%Y%m%d_%H%M%S}.pkl"
    _bundle = {'trial_df': trial_df, 'trial_summary': trial_summary, 'settings_df': settings_df, **extra_frames}
    with open(_path, 'wb') as _f:
        pickle.dump(_bundle, _f)
    print(f"Saved {list(_bundle.keys())} -> {_path}")
    return _path

print("Ready. Call save_analysis_pickle() to pickle trial_df / trial_summary / settings_df to disk,")
print("or save_analysis_pickle('my_analysis.pkl') to choose the filename.")

# Section 6e: H:M Ratio Summary

Box plot and histogram of H:M ratio per stimulation polarity group.
Shows distribution shape, standard deviation, CV, and 20th/80th percentiles.

In [ ]:
# ---- Stim polarity split ----
trials_by_polarity = split_trials_by_polarity(_plot_trials)

# ---- H:M Ratio Summary: box plot + histogram ----
# Uses the same M/H windows defined in Section 6a.
plot_hm_ratio_summary(
    trials_by_polarity, _plot_header,
    m_start_ms=M_START_MS, m_end_ms=M_END_MS,
    h_start_ms=H_START_MS, h_end_ms=H_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)
# ---- H-wave Regression: H-wave amplitude vs background EMG ----
plot_hwave_regression(
    _plot_trials, _plot_emg_blocks,
    m_start_ms=M_START_MS, m_end_ms=M_END_MS,
    h_start_ms=H_START_MS, h_end_ms=H_END_MS,
    sample_rate=recording_sample_rate or hrs1_header.sample_rate,
)

# Section 7: EMG Window Optimization

## Problem

Choose the best **EMG window** — defined by two decision variables:

| Decision variable | Options |
|---|---|
| **Window Location** (centre) | Q1, Median, or Q3 of the background EMG distribution |
| **Window Size** (EMG window width, ±hw µV) | narrow (±5 µV) → wide (±150 µV) |

## Objectives (simultaneously)

1. **Maximize trial rate** — a wider or better-placed window accepts more trials per session
2. **Minimize M-H variability** — a narrower window selects more uniform background states → more consistent M/H responses

These two objectives **conflict**: larger windows accept more trials but also let in more variable background states.

## Composite score

$$\text{score} = \alpha \times \text{trial\_score} + (1 - \alpha) \times \text{consistency\_score}$$

Both components are independently normalized to **[0, 1]** across all (centre, hw) pairs, so they are directly comparable regardless of units.

| α value | Priority |
|---|---|
| α = 1.0 | Maximize trials only |
| α = 0.5 | Balanced trade-off (default) |
| α = 0.0 | Minimize variability only |

## Pareto front

A (centre, hw) pair is **Pareto-optimal** if no other pair has *both* more trials *and* lower variability — i.e., you cannot improve one objective without hurting the other.
The Pareto front defines the set of efficient operating points.

## Outputs

- **Section 7a**: Compute scores + print ranked table and Pareto solutions
- **Section 7b**: Three-panel visualization:
  1. Pareto scatter — trial count vs variability (Pareto front highlighted)
  2. Composite score heatmap — quick visual of best (centre, hw) combinations
  3. Dual-axis trade-off lines — see exactly how trial count and variability change as you widen the window

In [ ]:
# ---- Section 7a: Compute optimization scores ----
#
# alpha: weight between trial rate and consistency
#   alpha = 1.0  -> maximize accepted trials only
#   alpha = 0.5  -> balanced trade-off (default)
#   alpha = 0.0  -> minimize variability only
#
# VARIABILITY_KEYS: which M-H metrics to include in the variability score
#   options: 'm_mra', 'h_mra', 'hm_ratio', 'm_size', 'h_size'
#
# VARIABILITY_STAT: 'cv' (coefficient of variation, dimensionless) or 'std'

ALPHA            = 0.5
VARIABILITY_KEYS = ['m_mra', 'h_mra']
VARIABILITY_STAT = 'cv'

opt_rows, opt_info = compute_optimization_scores(
    sweep_results, mh_variability,
    variability_keys=VARIABILITY_KEYS,
    variability_stat=VARIABILITY_STAT,
    alpha=ALPHA,
)
print_optimization_summary(opt_rows, opt_info, _plot_header, top_n=5)

In [ ]:
# ---- Section 7b: Optimization plots ----
# Pareto scatter, composite score heatmap, dual-axis trade-off lines.
# Re-run Section 7a with a different ALPHA to update the heatmap and ranking.S
plot_optimization(opt_rows, opt_info, _plot_header)